In [0]:
from pyspark.sql.types import *
from delta.tables import DeltaTable
from pyspark.sql.functions import *

In [0]:
%sql
create or replace table retail_sales_dw.config_table (
    pipeline_name string
  , file_path string
  , header string
  , delimiter string
  , table_name string
  , schema_detail map<string,string>
  , keys array<string>
  , write_mode string
)

In [0]:
def upsert_into(df:DataFrame,table_name:str,keys:list) -> DataFrame:
    delta_obj = DeltaTable.forName(spark,table_name)
    return (
        delta_obj.alias("t").merge(
            df.alias("s")," AND ".join([f"t.{key} = s.{key}" for key in keys])
            )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
        )

In [0]:
data = [{
    "pipeline_name": "shop_name",
    "file_path": "dbfs:/Volumes/workspace/retail_sales_dw/file_folder/shop_name_20260101.csv",
    "header": "true",
    "delimiter": ",",
    "table_name": "retail_sales_dw.shop_name",
    "schema_detail": {"shop_id": "int", "shop_name": "string", "branch_name": "string", "file_dt": "date"},
    "keys": ["shop_id"],
    "write_mode": "overwrite"
}]

schema = StructType([
    StructField("pipeline_name", StringType(), True),
    StructField("file_path", StringType(), True),
    StructField("header", StringType(), True),
    StructField("delimiter", StringType(), True),
    StructField("table_name", StringType(), True),
    StructField("schema_detail", MapType(StringType(), StringType()), True),
    StructField("keys", ArrayType(StringType()), True),
    StructField("write_mode", StringType(), True)
])

mock_df = spark.createDataFrame(data, schema)
upsert_into(mock_df,"retail_sales_dw.config_table",["pipeline_name"])

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, MapType, ArrayType

data = [{
    "pipeline_name": "fact_sales",
    "file_path": "/Volumes/workspace/retail_sales_dw/file_folder/fact_sales_20260101.parquet",
    "header": None,
    "delimiter":None,
    "table_name": "retail_sales_dw.fact_sales",
    "schema_detail": {"transaction_id": "long","shop_id": "long","sales_qty": "int","sales_amt": "decimal(18,2)","sales_date": "date"},
    "keys": ["transaction_id","shop_id"],
    "write_mode": "overwrite"
}]

schema = StructType([
    StructField("pipeline_name", StringType(), True),
    StructField("file_path", StringType(), True),
    StructField("header", StringType(), True),
    StructField("delimiter", StringType(), True),
    StructField("table_name", StringType(), True),
    StructField("schema_detail", MapType(StringType(), StringType()), True),
    StructField("keys", ArrayType(StringType()), True),
    StructField("write_mode", StringType(), True)
])

mock_df = spark.createDataFrame(data, schema)
upsert_into(mock_df,"retail_sales_dw.config_table",["pipeline_name"])

In [0]:
display(
    spark.table("retail_sales_dw.config_table")
    .filter("pipeline_name = 'fact_sales'")
)